# `test_nonlinear` — visual validation of Raman + Brillouin

**Taxonomy** (per `AGENTS.md`): a **visual test** (T-VISUAL) combined with two
**validation tests** (T-VALIDATION) against published data:

1. **Raman gain spectrum** plotted on top of the qualitative shape from Agrawal
   *Nonlinear Fiber Optics* 6th ed., Ch. 8 — peak at 13.2 THz, shoulder at 15.2 THz.
2. **Raman quantum-channel counts vs Ferreira da Silva et al.** (arXiv:1410.0656,
   Fig. 2): pump-wavelength-swept Stokes/anti-Stokes counts at a fixed quantum
   channel. WebPlotDigitizer points are embedded inline below.
3. **Brillouin gain profile** against Agrawal Ch. 9 + Kobyakov 2010 SMF-28 values.
4. Two **consistency tables** (T-PHYSICS / T-VALIDATION):
   - Raman peak g_R agreement across the three models.
   - Brillouin SBS threshold vs the Kobyakov 2010 review numbers.

**Location.** Intended for `BIFROST/test/human/test_nonlinear.ipynb`. Uses IJulia
(`Pkg.add("IJulia"); using IJulia; notebook()`) and the standard Plots.jl backend.


In [ ]:
# ---------- Includes ----------
using Plots, Printf, Statistics
gr()

# Core BIFROST stack
include(joinpath(@__DIR__, "..", "..", "src", "material-properties.jl"))
include(joinpath(@__DIR__, "..", "..", "src", "fiber",    "fiber-cross-section.jl"))
include(joinpath(@__DIR__, "..", "..", "src", "geometry", "path-geometry.jl"))
include(joinpath(@__DIR__, "..", "..", "src", "fiber",    "fiber-path.jl"))
include(joinpath(@__DIR__, "..", "..", "src", "fiber",    "fiber-path-modify.jl"))
include(joinpath(@__DIR__, "..", "..", "src", "path-integral.jl"))

# Nonlinear extension
include(joinpath(@__DIR__, "..", "..", "src", "nonlinear", "raman.jl"))
include(joinpath(@__DIR__, "..", "..", "src", "nonlinear", "brillouin.jl"))

# A reference SMF-28-like fiber used throughout this notebook.
const XS_SMF = FiberCrossSection(
    GermaniaSilicaGlass(0.036),   # 3.6 mol% GeO2 core
    GermaniaSilicaGlass(0.0),     # pure silica cladding
    8.2e-6, 125e-6;
    manufacturer = "Corning",
    model_number = "SMF-28-like",
)

const Λ_REF_M = 1550e-9
const T_REF_K = 297.15
const γ_SMF   = nonlinear_coefficient(XS_SMF, Λ_REF_M, T_REF_K)
const A_EFF   = effective_mode_area(XS_SMF, Λ_REF_M, T_REF_K)

println("Reference fiber: SMF-28-like, 3.6 mol% GeO2 core, 8.2 μm core diameter")
@printf("  γ (1550 nm, 297 K) = %.4e W⁻¹m⁻¹\n", γ_SMF)
@printf("  A_eff              = %.2f μm²\n", A_EFF*1e12)


## 1. Raman gain spectrum vs Agrawal NLFO Ch. 8

Agrawal's textbook reports for fused silica:
- Primary Raman peak at **13.2 THz** (Stokes shift)
- Secondary shoulder at **~15.2 THz**, ≈ 90 % of peak amplitude
- Material gain coefficient at peak: `g_R ≈ 1 × 10⁻¹³ m/W` at λ_p = 1 μm,
  scaling as 1/λ_p (so ≈ 6.5 × 10⁻¹⁴ m/W at 1.55 μm).

Our `g_R(Ω, γ)` returns the *fiber* gain coefficient `2γ·f_R·Im[h_R(Ω)]` with
units W⁻¹m⁻¹. To recover the material gain multiply by A_eff. We plot both
versions of all three models (`:bw`, `:hc`, `:tabulated`).


In [ ]:
freq_THz = collect(0.1:0.05:30)
Ω        = 2π .* freq_THz .* 1e12

models = [(:bw, "Blow-Wood", :firebrick), (:hc, "Hollenbeck-Cantrell", :seagreen),
          (:tabulated, "Lin & Agrawal tab.", :dodgerblue)]

# Left: fiber gain g_R (W⁻¹m⁻¹) — what our code returns directly
plt1 = plot(xlabel="Frequency shift Ω/2π  (THz)",
            ylabel="g_R  (×10⁻³ W⁻¹ m⁻¹)",
            title="Raman gain — fiber coefficient", legend=:topleft)
for (m, lbl, c) in models
    g = [Float64(g_R(o, γ_SMF; model=m)) for o in Ω]
    plot!(plt1, freq_THz, g .* 1e3, label=lbl, lw=2, color=c)
end
vline!(plt1, [13.2], ls=:dash, lc=:gray, label="13.2 THz peak (Agrawal)")
vline!(plt1, [15.2], ls=:dot,  lc=:gray, label="15.2 THz shoulder")

# Right: material gain g_R × A_eff (m/W) — comparable to Agrawal Table 8.1
plt2 = plot(xlabel="Frequency shift Ω/2π  (THz)",
            ylabel="g_R × A_eff   (×10⁻¹⁴ m/W)",
            title="Raman material gain — vs Agrawal Ch. 8", legend=:topleft)
for (m, lbl, c) in models
    g = [Float64(g_R(o, γ_SMF; model=m)) for o in Ω]
    plot!(plt2, freq_THz, (g .* A_EFF) .* 1e14, label=lbl, lw=2, color=c)
end
# Agrawal Table 8.1 reference points (rough)
agrawal_pts_THz = [13.2, 15.2]
agrawal_g_m_per_W = [6.5e-14, 5.8e-14]   # at 1.55 μm, after 1/λ scaling
scatter!(plt2, agrawal_pts_THz, agrawal_g_m_per_W .* 1e14;
         label="Agrawal NLFO Tab. 8.1", color=:black, marker=:square, ms=5)

plot(plt1, plt2, layout=(1,2), size=(1100, 400))


## 2. Raman quantum-channel counts vs Ferreira da Silva (2014)

Reproduces Fig. 2 of arXiv:1410.0656.

**Setup:** SMF-28, L = 7.5 km, P_pump = 1 mW, BW = 50 GHz, η = 15 %, τ = 2.5 ns,
α = 0.2 dB/km, γ = 1.3 × 10⁻³ W⁻¹m⁻¹, T = 300 K, λ_quantum = 1546.12 nm.
Counter-propagating geometry. Pump swept across the C+L band.

Stokes is selected when λ_pump < λ_quantum, anti-Stokes otherwise.


In [ ]:
# WebPlotDigitizer data from Fig. 2 of arXiv:1410.0656 (y-axis ×10⁻²)
const DASILVA_DATA = [
    1474.804 4.308; 1478.917 4.561; 1476.752 3.968; 1480.866 3.818;
    1482.815 3.842; 1485.196 3.842; 1486.928 3.392; 1488.877 3.258;
    1490.825 3.163; 1492.774 3.068; 1494.939 2.879; 1496.888 2.642;
    1499.053 3.013; 1500.785 2.642; 1502.950 2.918; 1504.899 2.666;
    1507.064 2.926; 1509.012 3.029; 1510.961 2.689; 1513.126 3.163;
    1514.858 2.950; 1516.806 3.226; 1518.972 3.518; 1520.920 3.368;
    1522.869 3.542; 1524.817 3.716; 1526.982 3.771; 1528.931 3.613;
    1530.880 3.866; 1532.828 4.024; 1534.993 3.858; 1536.942 3.195;
    1539.107 2.476; 1541.055 1.813; 1543.004 1.111; 1544.087 0.897;
    1544.953 1.142; 1546.901 1.150; 1548.200 1.079; 1549.066 0.929;
    1551.015 1.568; 1552.963 2.066; 1554.912 2.887; 1556.861 2.824;
    1559.026 2.982; 1560.974 3.132; 1563.139 2.879; 1565.304 2.879;
    1567.037 2.745; 1568.769 2.713; 1571.150 2.366; 1572.882 2.216;
    1575.047 1.987; 1577.212 1.947; 1579.161 1.592; 1581.110 1.687;
    1583.491 1.679; 1585.007 1.237; 1586.955 1.032; 1589.120 1.197;
    1591.069 1.047; 1593.234 1.205; 1594.966 1.197; 1597.131 1.047;
    1598.863 1.095; 1601.245 1.055; 1602.977 1.189; 1604.926 0.826;
    1606.874 0.739; 1609.039 1.118; 1611.204 1.134; 1612.936 1.126;
    1614.885 0.897; 1617.050 0.945; 1618.999 0.913; 1621.164 0.961;
    1623.329 0.929; 1625.061 1.047; 1627.009 1.095; 1628.742 0.945;
    1631.123 0.945; 1633.072 0.842; 1635.020 1.071; 1636.969 0.795;
    1638.917 0.668
]
paper_wl  = DASILVA_DATA[:,1]                  # nm
paper_cnt = DASILVA_DATA[:,2] .* 1e-2          # counts/trigger

# da Silva experimental constants
const LAMBDA_Q_M = 1546.12e-9
const GAMMA_DSV  = 1.3e-3                      # W⁻¹m⁻¹
const ALPHA_DSV  = 0.2 * log(10) / (10 * 1e3)  # 1/m, from 0.2 dB/km
const ETA_DSV    = 0.15
const TAU_DSV    = 2.5e-9
const BW_50G_M   = LAMBDA_Q_M^2 / 2.99792458e8 * 50e9   # 50 GHz in m at 1546 nm
const L_DSV      = 7.5e3

leff_back(L, α) = (1 - exp(-2α*L)) / (2α)

function counts_at(λp_m, model)
    sb = λp_m < LAMBDA_Q_M ? :stokes : :antistokes
    L_eff = leff_back(L_DSV, ALPHA_DSV)
    N = sprs_noise_in_channel(λp_m, LAMBDA_Q_M, BW_50G_M, 1e-3,
                                L_eff, GAMMA_DSV, 300.0;
                                sideband=sb, model=model)
    return N * TAU_DSV * ETA_DSV
end

# Compute counts for each model at every paper wavelength (skip ±0.5 nm near λ_q)
gap = abs.(paper_wl .- LAMBDA_Q_M*1e9) .< 0.5
results = Dict{Symbol, Vector{Float64}}()
for (m, _, _) in models
    cnts = Float64[]
    for (i, λnm) in enumerate(paper_wl)
        push!(cnts, gap[i] ? NaN : counts_at(λnm * 1e-9, m))
    end
    results[m] = cnts
end

plt = plot(xlabel="Pump wavelength (nm)",
           ylabel="Counts per trigger (×10⁻²)",
           title="Raman counts vs da Silva 2014 Fig. 2  (L=7.5 km, BW=50 GHz)",
           size=(1000, 500), legend=:topright)
for (m, lbl, c) in models
    plot!(plt, paper_wl, results[m] ./ 1e-2, label=lbl, lw=1.8, color=c)
end
scatter!(plt, paper_wl, paper_cnt ./ 1e-2,
         label="da Silva 2014 (Fig. 2)", color=:black, marker=:square, ms=4)
vline!(plt, [LAMBDA_Q_M*1e9], ls=:dot, lc=:gray, label="λ_q = 1546.12 nm")
annotate!(plt, [(1500, 4.5, ("Stokes side (pump bluer)", 9, :gray, :center)),
                (1600, 4.5, ("Anti-Stokes side (pump redder)", 9, :gray, :center))])
plt


## 3. Brillouin gain profile vs literature

Literature reference values for SMF-28 at 1550 nm:
- Brillouin shift `ν_B ≈ 11.0–11.2 GHz` (depends on GeO₂ doping, ~10.6 GHz for pure silica
  scaled up by acousto-optic coupling — Agrawal Ch. 9)
- FWHM linewidth `Δν_B ≈ 25–50 MHz` (varies with fiber)
- Peak material gain `g_B,peak ≈ 2.2 × 10⁻¹¹ m/W` for SMF-28
  (Niklès 1997 calibration at 3.6 mol% GeO₂)


In [ ]:
m_GeO2 = XS_SMF.core_material.x_ge

prof = brillouin_gain_profile(Λ_REF_M;
    freq_range_GHz=(-15, 15), N=4096, m_GeO2=m_GeO2)

freq = prof.freq_offset_GHz
g_B  = prof.g_B_mW
νB   = prof.nu_B_GHz
ΔνB  = prof.Gamma_B_MHz

plt = plot(freq, g_B .* 1e11, lw=2, color=:dodgerblue,
           xlabel="Frequency offset from pump (GHz)",
           ylabel="g_B  (×10⁻¹¹ m/W)",
           title=@sprintf("Brillouin gain — SMF-28-like  (νB = %.2f GHz, ΔνB ≈ %.0f MHz)", νB, ΔνB),
           size=(900, 400), legend=:topright)
vline!(plt, [-νB, νB], ls=:dash, lc=:firebrick, label="±ν_B")

# Literature reference band: ν_B = 11.0–11.2 GHz
vspan!(plt, [11.0, 11.2], color=:orange, alpha=0.15, label="Lit. ν_B band (Agrawal Ch. 9)")
vspan!(plt, [-11.2, -11.0], color=:orange, alpha=0.15, label="")
plt


## 4. Consistency Table A — Raman peak g_R across models

Internal consistency check: all three models should agree at the 13.2 THz peak
to within ~15 %. Larger disagreement would indicate a normalisation bug.


In [ ]:
Ω_peak = 2π * 13.2e12

println("Raman peak g_R at 13.2 THz, SMF-28-like fiber")
println("=" ^ 60)
@printf("%-25s  %15s  %15s\n", "Model", "g_R  (W⁻¹m⁻¹)", "g_R/A_eff (m/W)")
println("-" ^ 60)
peak_values = Float64[]
for (m, lbl, _) in models
    g = Float64(g_R(Ω_peak, γ_SMF; model=m))
    push!(peak_values, g)
    @printf("%-25s  %15.4e  %15.4e\n", lbl, g, g / A_EFF)
end
println("-" ^ 60)
spread = (maximum(peak_values) - minimum(peak_values)) / mean(peak_values)
@printf("Relative spread across models: %.1f %%   (expect < 15 %%)  %s\n",
        spread*100, spread < 0.15 ? "✓ PASS" : "✗ FAIL")


## 5. Consistency Table B — Brillouin threshold vs Kobyakov 2010

Smith (1972) threshold formula: `P_th = G_th · A_eff / (g_B · L_eff^back)`
with `G_th = 21`.

Kobyakov, Sauer & Chowdhury, *Adv. Opt. Photon.* **2**, 1 (2010), Table 3:
- 25 km SMF-28 at 1550 nm: P_th ≈ 5.5 mW
- 50 km SMF-28 at 1550 nm: P_th ≈ 4.8 mW
- 100 km SMF-28 at 1550 nm: P_th ≈ 4.4 mW (asymptotic, L_eff saturates)


In [ ]:
println("Brillouin SBS threshold vs Kobyakov 2010 (Table 3)")
println("=" ^ 65)
@printf("%-10s  %-10s  %-12s  %-12s  %-10s\n",
        "L (km)", "A_eff (μm²)", "P_th (mW)", "Lit. (mW)", "δ (%)")
println("-" ^ 65)
lit = Dict(25.0 => 5.5, 50.0 => 4.8, 100.0 => 4.4)
for L_km in (10.0, 25.0, 50.0, 100.0)
    th = brillouin_threshold(A_EFF, L_km*1e3;
                              λ_pump=Λ_REF_M,
                              g_B_peak=g_B_peak_GeO2(m_GeO2))
    P_mW = th.P_threshold_mW
    if haskey(lit, L_km)
        δ = 100 * (P_mW - lit[L_km]) / lit[L_km]
        @printf("%-10.0f  %-10.1f  %-12.2f  %-12.2f  %+10.1f\n",
                L_km, A_EFF*1e12, P_mW, lit[L_km], δ)
    else
        @printf("%-10.0f  %-10.1f  %-12.2f  %-12s  %-10s\n",
                L_km, A_EFF*1e12, P_mW, "—", "—")
    end
end
println("-" ^ 65)
println("Acceptable agreement: |δ| ≲ 30 %.  Kobyakov tabulates a wide range")
println("of P_th values across SMF-28 batches; tight match isn't expected.")


## 6. Brillouin shift vs GeO₂ doping (Niklès 1997)

Niklès, Thévenaz & Robert, *J. Lightwave Technol.* **15**, 1842 (1997) Table II:
- Pure SiO₂:     ν_B = 11.32 GHz at 1320 nm  ⇒  ~9.62 GHz at 1550 nm
- SMF-28 (3.6%): ν_B = 10.92 GHz at 1320 nm  ⇒  ~9.30 GHz at 1550 nm
- DSF (7.5%):    ν_B = 10.61 GHz at 1320 nm  ⇒  ~9.04 GHz at 1550 nm

Our `brillouin_freq_shift(λ; m_GeO2=x)` model scales the acoustic velocity by
`(1 + BRIL_VA_COEFF · x)` with `BRIL_VA_COEFF = -0.274`. The pattern should match.


In [ ]:
λ_test = 1550e-9
n_eff_test = 1.4447
println("Brillouin shift vs GeO₂ doping at 1550 nm")
println("=" ^ 50)
@printf("%-15s  %-12s  %-12s\n", "Fiber", "x_GeO2", "ν_B (GHz)")
println("-" ^ 50)
fibers = [("Pure SiO₂", 0.0),
          ("SMF-28",    0.036),
          ("DSF",       0.075)]
νs = Float64[]
for (name, x) in fibers
    ν = Float64(brillouin_freq_shift(λ_test; n_eff=n_eff_test, m_GeO2=x)) / 1e9
    push!(νs, ν)
    @printf("%-15s  %-12.3f  %-12.3f\n", name, x, ν)
end
println("-" ^ 50)
@printf("Shift trend (pure → 7.5%% GeO₂): %.2f → %.2f GHz, Δ = %.2f GHz\n",
        νs[1], νs[end], νs[1] - νs[end])
println("Niklès 1997 Table II (at 1320 nm, scaled): Δ ≈ 0.7 GHz across this range.")


## 7. Summary

| Test | What it checks | Expected |
|------|----------------|----------|
| 1    | Raman gain spectrum shape | Peak at 13.2 THz, shoulder at 15.2 THz, three models within ~15 % at peak |
| 2    | Raman counts vs da Silva 2014 Fig. 2 | Wavelength-swept Stokes/anti-Stokes shape matches paper data |
| 3    | Brillouin gain Lorentzian | Peak in 11.0–11.2 GHz band, FWHM 25–50 MHz |
| 4    | Internal Raman model agreement | < 15 % spread across BW / HC / Tabulated |
| 5    | Brillouin SBS threshold vs Kobyakov 2010 | within ~30 % across L = 25–100 km |
| 6    | Brillouin shift trend vs GeO₂ doping | Decreases monotonically with doping, Δ ≈ 0.7 GHz over 0–7.5 % |

If everything looks right, commit. If a plot looks off, prefer adjusting the
model parameters (constants in `raman.jl` / `brillouin.jl`) rather than
relaxing the comparison tolerances.
